# Homework 2

Target is fuel_efficiency_mpg. Features: engine_displacement, horsepower, vehicle_weight, model_year.

In [ ]:
import pandas as pd
import numpy as np

url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'
cols = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year', 'fuel_efficiency_mpg']
df = pd.read_csv(url)[cols].astype(float)
features = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']

In [ ]:
def split(seed):
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test
    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)
    return (df.iloc[idx[:n_train]],
            df.iloc[idx[n_train:n_train + n_val]],
            df.iloc[idx[n_train + n_val:]])


def prep(d, fill):
    d = d.copy()
    d['horsepower'] = d['horsepower'].fillna(fill)
    return d[features].values, d['fuel_efficiency_mpg'].values


def train(X, y, r=0.0):
    X = np.column_stack([np.ones(len(X)), X])
    XTX = X.T @ X + r * np.eye(X.shape[1])
    w = np.linalg.solve(XTX, X.T @ y)
    return w[0], w[1:]


def rmse(y, yp):
    return np.sqrt(((y - yp) ** 2).mean())


tr, va, te = split(42)

## Q1 — column with missing values

horsepower, 877 rows

In [ ]:
na = df.isna().sum()
na[na > 0]

## Q2 — median horsepower

254

In [ ]:
df['horsepower'].median()

## Q3 — fill with 0 or with the mean

2.205 with 0, 2.202 with the training mean. Mean wins, barely.

In [ ]:
out = {}
for label, fill in [('zero', 0.0), ('mean', tr['horsepower'].mean())]:
    Xtr, ytr = prep(tr, fill)
    Xva, yva = prep(va, fill)
    w0, w = train(Xtr, ytr)
    out[label] = round(rmse(yva, w0 + Xva @ w), 3)
out

## Q4 — best r

r = 0. The score only gets worse as r grows: 2.2053, 2.2058, 2.2241, 2.3492, 2.4094, 2.4195, 2.4292.

In [ ]:
Xtr, ytr = prep(tr, 0.0)
Xva, yva = prep(va, 0.0)
scores = {}
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0, w = train(Xtr, ytr, r)
    scores[r] = round(rmse(yva, w0 + Xva @ w), 4)
scores

## Q5 — std across seeds 0-9

0.029

In [ ]:
vals = []
for s in range(10):
    t, v, _ = split(s)
    Xt, yt = prep(t, 0.0)
    Xv, yv = prep(v, 0.0)
    w0, w = train(Xt, yt)
    vals.append(rmse(yv, w0 + Xv @ w))
round(float(np.std(vals)), 3)

## Q6 — test RMSE, seed 9, train+val, r=0.001

2.236

In [ ]:
t, v, te = split(9)
Xf, yf = prep(pd.concat([t, v]), 0.0)
Xte, yte = prep(te, 0.0)
w0, w = train(Xf, yf, 0.001)
rmse(yte, w0 + Xte @ w)